# 实战Kaggle比赛：狗的品种识别（ImageNet Dogs）

本节我们将在Kaggle上实战狗品种识别问题。
本次(**比赛网址是https://www.kaggle.com/c/dog-breed-identification**)。
 :numref:`fig_kaggle_dog`显示了鉴定比赛网页上的信息。
需要一个Kaggle账户才能提交结果。

在这场比赛中，我们将识别120类不同品种的狗。
这个数据集实际上是著名的ImageNet的数据集子集。与 :numref:`sec_kaggle_cifar10`中CIFAR-10数据集中的图像不同，
ImageNet数据集中的图像更高更宽，且尺寸不一。

![狗的品种鉴定比赛网站，可以通过单击“数据”选项卡来获得比赛数据集。](../img/kaggle-dog.jpg)
:width:`400px`
:label:`fig_kaggle_dog`


In [1]:
import os
import torch
import torchvision
from torch import nn
from d2l import torch as d2l

## 获取和整理数据集

比赛数据集分为训练集和测试集，分别包含RGB（彩色）通道的10222张、10357张JPEG图像。
在训练数据集中，有120种犬类，如拉布拉多、贵宾、腊肠、萨摩耶、哈士奇、吉娃娃和约克夏等。

### 下载数据集

登录Kaggle后，可以点击 :numref:`fig_kaggle_dog`中显示的竞争网页上的“数据”选项卡，然后点击“全部下载”按钮下载数据集。在`../data`中解压下载的文件后，将在以下路径中找到整个数据集：

* ../data/dog-breed-identification/labels.csv
* ../data/dog-breed-identification/sample_submission.csv
* ../data/dog-breed-identification/train
* ../data/dog-breed-identification/test


上述结构与 :numref:`sec_kaggle_cifar10`的CIFAR-10类似，其中文件夹`train/`和`test/`分别包含训练和测试狗图像，`labels.csv`包含训练图像的标签。

同样，为了便于入门，[**我们提供完整数据集的小规模样本**]：`train_valid_test_tiny.zip`。
如果要在Kaggle比赛中使用完整的数据集，则需要将下面的`demo`变量更改为`False`。


In [ ]:
#@save
d2l.DATA_HUB['dog_tiny'] = (d2l.DATA_URL + 'kaggle_dog_tiny.zip',
                            '0cb91d09b814ecdc07b50f31f8dcad3e81d6a86d')

# 如果使用Kaggle比赛的完整数据集，请将下面的变量更改为False
demo = True
if demo:
    data_dir = d2l.download_extract('dog_tiny')
else:
    data_dir = os.path.join('../pytorch', 'data', 'dog-breed-identification')

### [**整理数据集**]

我们可以像 :numref:`sec_kaggle_cifar10`中所做的那样整理数据集，即从原始训练集中拆分验证集，然后将图像移动到按标签分组的子文件夹中。

下面的`reorg_dog_data`函数读取训练数据标签、拆分验证集并整理训练集。


In [ ]:
#@save
def reorg_dog_data(data_dir, valid_ratio):
    """整理狗品种识别数据集，划分训练集、验证集和测试集"""

    # 读取CSV标签，建立图片名称与狗品种的映射关系
    labels = d2l.read_csv_labels(os.path.join(data_dir, 'labels.csv'))

    # 按指定比例划分训练集和验证集
    d2l.reorg_train_valid(data_dir, labels, valid_ratio)

    # 整理无标签测试集，统一放入unknown目录
    d2l.reorg_test(data_dir)


# 演示模式使用32，完整训练模式使用128
batch_size = 32 if demo else 128

valid_ratio = 0.1  # 设置验证集划分比例为10%

# 执行狗品种识别数据集的整理
reorg_dog_data(data_dir, valid_ratio)

## [**图像增广**]

回想一下，这个狗品种数据集是ImageNet数据集的子集，其图像大于 :numref:`sec_kaggle_cifar10`中CIFAR-10数据集的图像。
下面我们看一下如何在相对较大的图像上使用图像增广。


In [ ]:
# 定义狗品种识别训练集的数据增强和预处理流程
transform_train = torchvision.transforms.Compose([

    # 随机裁剪原图8%～100%的区域，宽高比在3/4～4/3之间
    # 将裁剪结果缩放为224×224
    torchvision.transforms.RandomResizedCrop(
        224, scale=(0.08, 1.0), ratio=(3.0/4.0, 4.0/3.0)),

    # 以50%的概率水平翻转图像
    torchvision.transforms.RandomHorizontalFlip(),

    # 随机调整亮度、对比度和饱和度，增强颜色多样性
    torchvision.transforms.ColorJitter(
        brightness=0.4, contrast=0.4, saturation=0.4),

    # 将图像转换为张量，像素值缩放到[0,1]
    torchvision.transforms.ToTensor(),

    # 使用ImageNet的RGB通道均值和标准差进行标准化
    torchvision.transforms.Normalize(
        [0.485, 0.456, 0.406],  # RGB均值
        [0.229, 0.224, 0.225]) # RGB标准差
])

测试时，我们只使用确定性的图像预处理操作。


In [ ]:
# 定义狗品种识别测试集的图像预处理流程
transform_test = torchvision.transforms.Compose([

    # 缩放图像，使较短边达到256像素，保持宽高比
    torchvision.transforms.Resize(256),

    # 从图像中心裁剪224×224的区域
    torchvision.transforms.CenterCrop(224),

    # 转换为PyTorch张量，像素值缩放到[0,1]
    torchvision.transforms.ToTensor(),

    # 使用ImageNet的RGB通道均值和标准差进行标准化
    torchvision.transforms.Normalize(
        [0.485, 0.456, 0.406],   # RGB均值
        [0.229, 0.224, 0.225])  # RGB标准差
])

## [**读取数据集**]

与 :numref:`sec_kaggle_cifar10`一样，我们可以读取整理后的含原始图像文件的数据集。


In [ ]:
# 加载训练集和完整训练集，使用随机数据增强
train_ds, train_valid_ds = [
    torchvision.datasets.ImageFolder(
        os.path.join(data_dir, 'train_valid_test', folder),  # 数据集路径
        transform=transform_train  # 随机裁剪、翻转、颜色增强和标准化
    )
    for folder in ['train', 'train_valid']
]

# 加载验证集和测试集，使用固定预处理
valid_ds, test_ds = [
    torchvision.datasets.ImageFolder(
        os.path.join(data_dir, 'train_valid_test', folder),  # 数据集路径
        transform=transform_test  # 缩放、中心裁剪和标准化
    )
    for folder in ['valid', 'test']
]

下面我们创建数据加载器实例的方式与 :numref:`sec_kaggle_cifar10`相同。


In [ ]:
# 创建训练集和完整训练集的数据迭代器
train_iter, train_valid_iter = [
    torch.utils.data.DataLoader(
        dataset,          # 数据集对象
        batch_size,       # 每批加载的图片数量
        shuffle=True,     # 每轮训练前随机打乱数据
        drop_last=True    # 丢弃最后不足一个批次的数据
    )
    for dataset in (train_ds, train_valid_ds)
]

# 创建验证集数据迭代器
valid_iter = torch.utils.data.DataLoader(
    valid_ds,             # 验证集
    batch_size,           # 批量大小
    shuffle=False,        # 不打乱样本顺序
    drop_last=True        # 丢弃最后不完整的批次
)

# 创建测试集数据迭代器
test_iter = torch.utils.data.DataLoader(
    test_ds,              # 测试集
    batch_size,           # 批量大小
    shuffle=False,        # 保持顺序，方便匹配图片ID与预测结果
    drop_last=False       # 保留全部测试样本
)

## [**微调预训练模型**]

同样，本次比赛的数据集是ImageNet数据集的子集。
因此，我们可以使用 :numref:`sec_fine_tuning`中讨论的方法在完整ImageNet数据集上选择预训练的模型，然后使用该模型提取图像特征，以便将其输入到定制的小规模输出网络中。
深度学习框架的高级API提供了在ImageNet数据集上预训练的各种模型。
在这里，我们选择预训练的ResNet-34模型，我们只需重复使用此模型的输出层（即提取的特征）的输入。
然后，我们可以用一个可以训练的小型自定义输出网络替换原始输出层，例如堆叠两个完全连接的图层。
与 :numref:`sec_fine_tuning`中的实验不同，以下内容不重新训练用于特征提取的预训练模型，这节省了梯度下降的时间和内存空间。

回想一下，我们使用三个RGB通道的均值和标准差来对完整的ImageNet数据集进行图像标准化。
事实上，这也符合ImageNet上预训练模型的标准化操作。


In [ ]:
def get_net(devices):
    finetune_net = nn.Sequential()
    finetune_net.features = torchvision.models.resnet34(pretrained=True)
    # 定义一个新的输出网络，共有120个输出类别
    finetune_net.output_new = nn.Sequential(nn.Linear(1000, 256),
                                            nn.ReLU(),
                                            nn.Linear(256, 120))
    # 将模型参数分配给用于计算的CPU或GPU
    finetune_net = finetune_net.to(devices[0])
    # 冻结参数
    for param in finetune_net.features.parameters():
        param.requires_grad = False
    return finetune_net

在[**计算损失**]之前，我们首先获取预训练模型的输出层的输入，即提取的特征。
然后我们使用此特征作为我们小型自定义输出网络的输入来计算损失。


In [ ]:
loss = nn.CrossEntropyLoss(reduction='none')

def evaluate_loss(data_iter, net, devices):
    l_sum, n = 0.0, 0
    for features, labels in data_iter:
        features, labels = features.to(devices[0]), labels.to(devices[0])
        outputs = net(features)
        l = loss(outputs, labels)
        l_sum += l.sum()
        n += labels.numel()
    return (l_sum / n).to('cpu')

## 定义[**训练函数**]

我们将根据模型在验证集上的表现选择模型并调整超参数。
模型训练函数`train`只迭代小型自定义输出网络的参数。


In [ ]:
def train(net, train_iter, valid_iter, num_epochs, lr, wd, devices,
          lr_period, lr_decay):
    """训练狗品种识别模型，支持多GPU训练和学习率衰减"""

    # 使用DataParallel实现多GPU并行训练
    # 将模型移动到主GPU
    net = nn.DataParallel(net, device_ids=devices).to(devices[0])

    # 创建SGD优化器，只更新需要计算梯度的参数
    trainer = torch.optim.SGD(
        (param for param in net.parameters() if param.requires_grad),
        lr=lr,               # 初始学习率
        momentum=0.9,        # 动量系数
        weight_decay=wd     # 权重衰减，防止过拟合
    )

    # 每隔lr_period轮，将学习率乘以lr_decay
    scheduler = torch.optim.lr_scheduler.StepLR(
        trainer, lr_period, lr_decay
    )

    # 获取每轮训练的批次数，并创建计时器
    num_batches, timer = len(train_iter), d2l.Timer()

    # 设置训练损失曲线的图例
    legend = ['train loss']

    # 如果提供验证集，则同时绘制验证损失
    if valid_iter is not None:
        legend.append('valid loss')

    # 创建动态绘图工具
    animator = d2l.Animator(
        xlabel='epoch',
        xlim=[1, num_epochs],
        legend=legend
    )

    # 开始训练，共num_epochs轮
    for epoch in range(num_epochs):

        # 切换到训练模式
        net.train()

        # 累加训练损失总和和样本数量
        metric = d2l.Accumulator(2)

        # 按批次读取训练数据
        for i, (features, labels) in enumerate(train_iter):

            timer.start()  # 开始计时

            # 将图像和标签移动到主GPU
            features = features.to(devices[0])
            labels = labels.to(devices[0])

            # 清空上一批次的梯度
            trainer.zero_grad()

            # 前向传播：计算模型输出
            output = net(features)

            # 计算当前批次的损失
            l = loss(output, labels).sum()

            # 反向传播：计算参数梯度
            l.backward()

            # 更新可训练参数
            trainer.step()

            # 累加损失与样本数量
            metric.add(l, labels.shape[0])

            timer.stop()  # 结束计时

            # 每轮大约绘制5次训练损失
            if (i + 1) % max(1, num_batches // 5) == 0 \
                    or i == num_batches - 1:
                animator.add(
                    epoch + (i + 1) / num_batches,
                    (metric[0] / metric[1], None)
                )

        # 记录当前轮的平均训练损失
        measures = f'train loss {metric[0] / metric[1]:.3f}'

        # 如果提供验证集，则计算验证损失
        if valid_iter is not None:
            valid_loss = evaluate_loss(valid_iter, net, devices)

            # 绘制验证损失曲线
            animator.add(
                epoch + 1,
                (None, valid_loss.detach().cpu())
            )

        # 更新学习率
        scheduler.step()

    # 输出最后一轮的验证损失
    if valid_iter is not None:
        measures += f', valid loss {valid_loss:.3f}'

    # 输出训练损失和训练速度
    print(
        measures +
        f'\n{metric[1] * num_epochs / timer.sum():.1f}'
        f' examples/sec on {str(devices)}'
    )

## [**训练和验证模型**]

现在我们可以训练和验证模型了，以下超参数都是可调的。
例如，我们可以增加迭代轮数。
另外，由于`lr_period`和`lr_decay`分别设置为2和0.9，
因此优化算法的学习速率将在每2个迭代后乘以0.9。


In [ ]:
# 获取所有可用GPU，设置训练轮数、学习率和权重衰减
devices, num_epochs, lr, wd = (
    d2l.try_all_gpus(),  # 获取所有可用GPU，没有GPU则使用CPU
    10,                  # 训练10轮（Epoch）
    1e-4,                # 初始学习率：0.0001
    1e-4                 # 权重衰减：0.0001，缓解过拟合
)

# 设置学习率调度参数，并创建狗品种识别模型
lr_period, lr_decay, net = (
    2,                  # 每2轮调整一次学习率
    0.9,                # 每次将学习率乘以0.9
    get_net(devices)    # 创建模型，通常使用预训练网络
)

# 开始训练模型，并使用验证集评估训练效果
train(
    net,                # 神经网络模型
    train_iter,         # 训练集数据迭代器
    valid_iter,         # 验证集数据迭代器
    num_epochs,         # 训练轮数
    lr,                 # 初始学习率
    wd,                 # 权重衰减
    devices,            # 训练设备
    lr_period,          # 学习率调整周期
    lr_decay            # 学习率衰减系数
)

## [**对测试集分类**]并在Kaggle提交结果

与 :numref:`sec_kaggle_cifar10`中的最后一步类似，最终所有标记的数据（包括验证集）都用于训练模型和对测试集进行分类。
我们将使用训练好的自定义输出网络进行分类。


In [ ]:
# 重新创建模型，准备使用全部有标签数据进行训练
net = get_net(devices)

# 使用完整训练集（训练集+验证集）训练最终模型
# valid_iter=None：不再进行验证，直接训练最终提交模型
train(net, train_valid_iter, None, num_epochs, lr, wd, devices,
      lr_period, lr_decay)

# 保存测试集所有图片的预测概率
preds = []

# 切换到评估模式，关闭Dropout等训练行为
net.eval()

# 测试阶段不需要计算梯度，减少显存占用
with torch.no_grad():

    # 按批次读取测试图片，label为占位标签，不参与预测
    for data, label in test_iter:

        # 将图片移动到GPU，进行前向传播
        # softmax将模型输出转换为各狗品种的预测概率
        output = torch.nn.functional.softmax(
            net(data.to(devices[0])), dim=1
        )

        # 将预测概率从GPU移动到CPU，转换为NumPy数组
        preds.extend(output.cpu().numpy())

# 获取测试图片文件名，并按字符串顺序排序
ids = sorted(os.listdir(
    os.path.join(data_dir, 'train_valid_test', 'test', 'unknown')
))

# 创建Kaggle提交文件
with open('submission.csv', 'w') as f:

    # 写入表头：图片ID + 所有狗品种名称
    f.write('id,' + ','.join(train_valid_ds.classes) + '\n')

    # 遍历图片ID及对应的预测概率
    for i, output in zip(ids, preds):

        # 去掉图片扩展名，获取图片ID
        # 将120个类别的预测概率写入CSV
        f.write(
            i.split('.')[0] + ',' +
            ','.join([str(num) for num in output]) + '\n'
        )

上面的代码将生成一个`submission.csv`文件，以 :numref:`sec_kaggle_house`中描述的方式提在Kaggle上提交。

## 小结

* ImageNet数据集中的图像比CIFAR-10图像尺寸大，我们可能会修改不同数据集上任务的图像增广操作。
* 要对ImageNet数据集的子集进行分类，我们可以利用完整ImageNet数据集上的预训练模型来提取特征并仅训练小型自定义输出网络，这将减少计算时间和节省内存空间。